# Learning to Unblend the Sky

This notebook runs the controlled deblending experiment: load Galaxy10 DECaLS images, split original images before blending, generate synthetic blends, compare simple baselines, train a compact U-Net, and inspect reconstructions by blend difficulty.

## 1. Setup / Imports

In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import baselines as gd_baselines
from src import blend as gd_blend
from src import data as gd_data
from src import models as gd_models
from src import train as gd_train
from src import utils as gd_utils

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
rng = np.random.default_rng(seed)

device = gd_train.resolve_device("auto")
print(f"Project root: {PROJECT_ROOT}")
print(f"Using device: {device}")

## 2. Config

In [ ]:
DATASET_PATH = PROJECT_ROOT / "data" / "Galaxy10_DECals.h5"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

config = {
    "dataset_path": str(DATASET_PATH),
    "train_frac": 0.70,
    "val_frac": 0.15,
    "test_frac": 0.15,
    "n_train_blends": 500,
    "n_val_blends": 100,
    "n_test_blends": 100,
    "subset_train_images": 500,
    "subset_val_images": 100,
    "subset_test_images": 100,
    "blend_params": {
        "max_shift": 56,
        "brightness_range": (0.5, 1.2),
        "blur_range": (0.0, 0.3),
        "noise_range": (0.0, 0.01),
        "rotation_range": (0.0, 0.0),
    },
    "model_params": {
        "in_channels": 3,
        "out_channels": 3,
        "base_channels": 32,
        "norm": True,
    },
    "training_params": {
        "batch_size": 8,
        "num_epochs": 10,
        "learning_rate": 1e-3,
        "weight_decay": 0.0,
        "checkpoint_path": str(OUTPUT_DIR / "unet_checkpoint.pth"),
    },
}

OUTPUT_DIR.mkdir(exist_ok=True)
config

## 3. Load Dataset

In [ ]:
images_raw, labels, metadata = gd_data.load_galaxy10(config["dataset_path"])
images = gd_data.normalise_images(images_raw)

(X_train, y_train), (X_val, y_val), (X_test, y_test) = gd_data.split_dataset(
    images,
    labels,
    train_frac=config["train_frac"],
    val_frac=config["val_frac"],
    test_frac=config["test_frac"],
    seed=seed,
)

print(f"Loaded {len(images)} images")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Metadata fields: {sorted(metadata.keys())}")

## 4. Inspect Original Images

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for idx, ax in enumerate(axes):
    ax.imshow(X_train[idx])
    ax.set_title(f"label={int(y_train[idx])}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Generate Synthetic Blends

If `src/blend.py` is edited during an active Jupyter session, restart the kernel or explicitly reload the module and regenerate `blends_train`, `blends_val`, and `blends_test`. Existing blend objects remain in memory and will not update automatically.

In [ ]:
X_train_small = X_train[: config["subset_train_images"]]
X_val_small = X_val[: config["subset_val_images"]]
X_test_small = X_test[: config["subset_test_images"]]

blend_params = config["blend_params"]

blends_train = gd_blend.generate_blends(
    X_train_small,
    n_blends=config["n_train_blends"],
    max_shift=blend_params["max_shift"],
    brightness_range=blend_params["brightness_range"],
    blur_range=blend_params["blur_range"],
    noise_range=blend_params["noise_range"],
    rotation_range=blend_params["rotation_range"],
    rng=rng,
)

blends_val = gd_blend.generate_blends(
    X_val_small,
    n_blends=config["n_val_blends"],
    max_shift=blend_params["max_shift"],
    brightness_range=blend_params["brightness_range"],
    blur_range=blend_params["blur_range"],
    noise_range=blend_params["noise_range"],
    rotation_range=blend_params["rotation_range"],
    rng=rng,
)

blends_test = gd_blend.generate_blends(
    X_test_small,
    n_blends=config["n_test_blends"],
    max_shift=blend_params["max_shift"],
    brightness_range=blend_params["brightness_range"],
    blur_range=blend_params["blur_range"],
    noise_range=blend_params["noise_range"],
    rotation_range=blend_params["rotation_range"],
    rng=rng,
)

print(f"Train blends: {len(blends_train)}")
print(f"Val blends: {len(blends_val)}")
print(f"Test blends: {len(blends_test)}")

## 6. Visualize Blends

In [ ]:
sample_indices = rng.choice(len(blends_train), size=min(3, len(blends_train)), replace=False)
fig, axes = plt.subplots(len(sample_indices), 3, figsize=(9, 3 * len(sample_indices)))
if len(sample_indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, idx in enumerate(sample_indices):
    sample = blends_train[int(idx)]
    axes[row, 0].imshow(sample["target"])
    axes[row, 0].set_title("Target")
    axes[row, 1].imshow(sample["contaminant"])
    axes[row, 1].set_title("Contaminant")
    axes[row, 2].imshow(sample["blended"])
    axes[row, 2].set_title(f"Blend: {sample['info']['difficulty']}")
    for col in range(3):
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 7. Baseline Evaluation

In [ ]:
num_eval_samples = min(50, len(blends_test))
samples = blends_test[:num_eval_samples]

targets = [sample["target"] for sample in samples]
identity_outputs = [gd_baselines.identity_baseline(sample["blended"]) for sample in samples]
threshold_outputs = [gd_baselines.threshold_baseline(sample["blended"]) for sample in samples]

identity_metrics = gd_utils.compute_metrics(identity_outputs, targets)
threshold_metrics = gd_utils.compute_metrics(threshold_outputs, targets)

print("Identity baseline:", identity_metrics)
print("Threshold baseline:", threshold_metrics)

## 8. Train Model

In [ ]:
train_dataset = gd_train.BlendDataset(blends_train)
val_dataset = gd_train.BlendDataset(blends_val)

model = gd_models.UNet(**config["model_params"]).to(device)

train_losses, val_losses = gd_train.train_model(
    model,
    train_dataset,
    val_dataset,
    num_epochs=config["training_params"]["num_epochs"],
    batch_size=config["training_params"]["batch_size"],
    learning_rate=config["training_params"]["learning_rate"],
    weight_decay=config["training_params"]["weight_decay"],
    device=device,
    checkpoint_path=config["training_params"]["checkpoint_path"],
)

plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Evaluate Model

In [ ]:
test_dataset = gd_train.BlendDataset(blends_test)
test_metrics, reconstructions = gd_train.evaluate_model(
    model,
    test_dataset,
    device=device,
)

test_metrics

## 10. Visualize Reconstructions

In [ ]:
num_show = min(3, len(blends_test))
fig, axes = plt.subplots(num_show, 4, figsize=(12, 3 * num_show))
if num_show == 1:
    axes = np.expand_dims(axes, axis=0)

for idx in range(num_show):
    sample = blends_test[idx]
    axes[idx, 0].imshow(sample["target"])
    axes[idx, 0].set_title("Target")
    axes[idx, 1].imshow(sample["contaminant"])
    axes[idx, 1].set_title("Contaminant")
    axes[idx, 2].imshow(sample["blended"])
    axes[idx, 2].set_title(f"Blend: {sample['info']['difficulty']}")
    axes[idx, 3].imshow(np.clip(reconstructions[idx], 0.0, 1.0))
    axes[idx, 3].set_title("Reconstruction")
    for col in range(4):
        axes[idx, col].axis("off")

plt.tight_layout()
plt.show()

## 11. Difficulty Analysis

In [ ]:
rows = []
for sample, reconstruction in zip(blends_test, reconstructions):
    values = gd_utils.compute_metrics(reconstruction, sample["target"])
    rows.append({"difficulty": sample["info"]["difficulty"], **values})

by_difficulty = {}
for difficulty in sorted({row["difficulty"] for row in rows}):
    group = [row for row in rows if row["difficulty"] == difficulty]
    by_difficulty[difficulty] = {
        metric: float(np.mean([row[metric] for row in group]))
        for metric in ("mse", "mae", "psnr", "ssim")
    }

by_difficulty

## 12. Notes for Scaling Up / Rotation Stress Test

For larger experiments, increase the subset sizes and blend counts after confirming memory use. Keep train, validation, and test blends generated from separate original-image splits.

Rotation is disabled by default. To stress-test orientation robustness, set `config["blend_params"]["rotation_range"] = (0.0, 180.0)`, regenerate all blend sets, and visually inspect examples before training. Rotation should be treated as an experimental condition because interpolation artifacts can affect evaluation.